# AirOps 360 - Bronze Environment Setup

## Working convention

- GitHub is the source of truth for code, documentation, contracts, and versioned configuration.
- Microsoft Fabric is the execution environment for pipelines, notebooks, Lakehouses, and operational runs.
- `config/airports_v0.1.csv` is governed in GitHub and copied into the Bronze Lakehouse for runtime use.
- Raw source files are preserved under `Files/raw`.
- Managed Bronze Delta representations use the `brz_` prefix.
- Silver and Gold transformations are not performed in this notebook.
- This setup notebook is safe to rerun and must not overwrite populated Bronze tables.

In [1]:
from datetime import datetime, timezone

print("=" * 70)
print("AirOps 360 - Bronze Environment Setup")
print("=" * 70)
print("UTC:", datetime.now(timezone.utc).isoformat())

required_paths = [
    "Files/raw/flights/year=2026/month=04",
    "Files/raw/weather",
    "Files/reference/airports",
]

for path in required_paths:
    notebookutils.fs.mkdirs(path)
    print(f"READY: {path}")

print("\nFolder setup complete.")

StatementMeta(, 453adbe2-d7fb-429b-84e4-7be78494f99f, 3, Finished, Available, Finished, False)

AirOps 360 - Bronze Environment Setup
UTC: 2026-09-10T05:16:27.269637+00:00
READY: Files/raw/flights/year=2026/month=04
READY: Files/raw/weather
READY: Files/reference/airports

Folder setup complete.


In [3]:
verification_paths = [
    "Files/raw",
    "Files/raw/flights",
    "Files/raw/flights/year=2026/month=04",
    "Files/raw/weather",
    "Files/reference",
    "Files/reference/airports",
]

print("AIROPS 360 DIRECTORY VERIFICATION")
print("-" * 70)

for path in verification_paths:
    try:
        notebookutils.fs.ls(path)
        print(f"PASS: {path}")
    except Exception as exc:
        print(f"FAIL: {path} -> {exc}")
        raise

StatementMeta(, 453adbe2-d7fb-429b-84e4-7be78494f99f, 5, Finished, Available, Finished, False)

AIROPS 360 DIRECTORY VERIFICATION
----------------------------------------------------------------------
PASS: Files/raw
PASS: Files/raw/flights
PASS: Files/raw/flights/year=2026/month=04
PASS: Files/raw/weather
PASS: Files/reference
PASS: Files/reference/airports


In [4]:
config_path = "Files/reference/airports/airports_v0.1.csv"

airport_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(config_path)
)

expected_columns = {
    "rank",
    "iata_code",
    "airport_name",
    "latitude",
    "longitude",
    "iana_timezone",
    "active",
}

actual_columns = set(airport_df.columns)

airport_count = airport_df.count()
active_count = airport_df.filter("active = true").count()

print("Airport rows:", airport_count)
print("Active airports:", active_count)
print("Columns:", airport_df.columns)

assert airport_count == 15, f"Expected 15 airports, found {airport_count}"
assert active_count == 15, f"Expected 15 active airports, found {active_count}"
assert expected_columns == actual_columns, (
    f"Column mismatch. Expected {expected_columns}, got {actual_columns}"
)

print("\nPASS: airports_v0.1.csv matches AirOps 360 configuration contract.")

StatementMeta(, 453adbe2-d7fb-429b-84e4-7be78494f99f, 6, Finished, Available, Finished, False)

Airport rows: 15
Active airports: 15
Columns: ['rank', 'iata_code', 'airport_name', 'latitude', 'longitude', 'iana_timezone', 'active']

PASS: airports_v0.1.csv matches AirOps 360 configuration contract.


In [5]:
bts_path = (
    "Files/raw/flights/year=2026/month=04/"
    "bts_reporting_carrier_ontime_2026_04.csv"
)

bts_source = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(bts_path)
)

required_fields = {
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "CRSArrTime",
    "ArrTime",
    "ArrDelay",
    "Cancelled",
    "Diverted",
    "AirTime",
    "Distance",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
}

source_columns = bts_source.columns
missing_fields = required_fields - set(source_columns)

print("BTS raw source validation")
print("-" * 60)
print("Source columns:", len(source_columns))
print("Required fields present:", len(required_fields - missing_fields))
print("Missing required fields:", sorted(missing_fields))
print("First 10 columns:", source_columns[:10])
print("Last 5 columns:", source_columns[-5:])

assert len(source_columns) == 110, (
    f"Expected 110 source columns, found {len(source_columns)}"
)

assert not missing_fields, (
    f"Missing required fields: {sorted(missing_fields)}"
)

print("\nPASS: April BTS raw source matches the profiled source contract.")

StatementMeta(, 453adbe2-d7fb-429b-84e4-7be78494f99f, 7, Finished, Available, Finished, False)

BTS raw source validation
------------------------------------------------------------
Source columns: 110
Required fields present: 20
Missing required fields: []
First 10 columns: ['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number']
Last 5 columns: ['Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum', '_c109']

PASS: April BTS raw source matches the profiled source contract.


In [6]:
from pyspark.sql import functions as F

if not spark.catalog.tableExists("brz_bts_flights"):

    bronze_shell = (
        bts_source
        .limit(0)
        .withColumn("_bronze_run_id", F.lit(None).cast("string"))
        .withColumn("_bronze_load_id", F.lit(None).cast("string"))
        .withColumn("_bronze_batch_key", F.lit(None).cast("string"))
        .withColumn("_bronze_contract_version", F.lit(None).cast("string"))
        .withColumn("_bronze_source_name", F.lit(None).cast("string"))
        .withColumn("_bronze_source_file_name", F.lit(None).cast("string"))
        .withColumn("_bronze_source_hash", F.lit(None).cast("string"))
        .withColumn("_bronze_ingested_at_utc", F.lit(None).cast("timestamp"))
        .withColumn("_bronze_load_year", F.lit(None).cast("int"))
        .withColumn("_bronze_load_month", F.lit(None).cast("int"))
    )

    (
        bronze_shell.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("brz_bts_flights")
    )

    print("CREATED: brz_bts_flights")

else:
    print("EXISTS: brz_bts_flights - left unchanged")

print(
    "Rows:",
    spark.table("brz_bts_flights").count()
)

print(
    "Columns:",
    len(spark.table("brz_bts_flights").columns)
)

StatementMeta(, 453adbe2-d7fb-429b-84e4-7be78494f99f, 8, Finished, Available, Finished, False)

CREATED: brz_bts_flights
Rows: 0
Columns: 120


In [7]:
spark.sql("""
CREATE TABLE IF NOT EXISTS brz_weather_api_raw (
    response_json STRING,
    airport_code STRING,
    request_start_date DATE,
    request_end_date DATE,
    response_timezone STRING,
    response_utc_offset_seconds INT,

    _bronze_run_id STRING,
    _bronze_load_id STRING,
    _bronze_batch_key STRING,
    _bronze_contract_version STRING,
    _bronze_source_name STRING,
    _bronze_ingested_at_utc TIMESTAMP
)
USING DELTA
""")

print("READY: brz_weather_api_raw")
print("Rows:", spark.table("brz_weather_api_raw").count())
print("Columns:", len(spark.table("brz_weather_api_raw").columns))

StatementMeta(, 453adbe2-d7fb-429b-84e4-7be78494f99f, 9, Finished, Available, Finished, False)

READY: brz_weather_api_raw
Rows: 0
Columns: 12


In [8]:
spark.sql("""
CREATE TABLE IF NOT EXISTS brz_ingestion_audit (
    run_id STRING,
    load_id STRING,
    attempt_no INT,
    batch_key STRING,
    contract_version STRING,

    source_name STRING,
    source_object STRING,
    source_uri STRING,
    source_file_name STRING,
    source_hash STRING,

    load_year INT,
    load_month INT,
    airport_code STRING,
    variable_set_version STRING,

    pipeline_name STRING,
    activity_name STRING,
    load_mode STRING,

    started_at_utc TIMESTAMP,
    completed_at_utc TIMESTAMP,
    ingestion_status STRING,

    source_row_count BIGINT,
    source_column_count INT,
    processed_row_count BIGINT,

    target_row_count_before BIGINT,
    target_row_count_after BIGINT,
    rejected_row_count BIGINT,

    reconciliation_status STRING,
    parameters_json STRING,

    error_class STRING,
    error_message STRING
)
USING DELTA
""")

print("READY: brz_ingestion_audit")
print("Rows:", spark.table("brz_ingestion_audit").count())
print("Columns:", len(spark.table("brz_ingestion_audit").columns))

StatementMeta(, 453adbe2-d7fb-429b-84e4-7be78494f99f, 10, Finished, Available, Finished, False)

READY: brz_ingestion_audit
Rows: 0
Columns: 30


In [9]:
print("=" * 72)
print("AIROPS 360 - TASK 8 BRONZE ENVIRONMENT VALIDATION")
print("=" * 72)

all_passed = True

expected_tables = {
    "brz_bts_flights": 120,
    "brz_weather_api_raw": 12,
    "brz_ingestion_audit": 30,
}

print("\n1. DELTA TABLES")
print("-" * 72)

for table_name, expected_column_count in expected_tables.items():

    if not spark.catalog.tableExists(table_name):
        print(f"FAIL: {table_name} does not exist")
        all_passed = False
        continue

    df = spark.table(table_name)
    row_count = df.count()
    column_count = len(df.columns)

    print(
        f"{table_name:<28} "
        f"rows={row_count:<4} "
        f"columns={column_count}"
    )

    if row_count != 0:
        print(
            f"FAIL: {table_name} should still contain 0 rows "
            f"during Task 8."
        )
        all_passed = False

    if column_count != expected_column_count:
        print(
            f"FAIL: expected {expected_column_count} columns, "
            f"found {column_count}"
        )
        all_passed = False


print("\n2. AIRPORT CONFIGURATION")
print("-" * 72)

airport_rows = airport_df.count()
active_airports = airport_df.filter("active = true").count()

print("Airport rows:", airport_rows)
print("Active airports:", active_airports)

if airport_rows != 15 or active_airports != 15:
    print("FAIL: airport configuration does not match v0.1")
    all_passed = False
else:
    print("PASS: airport configuration")


print("\n3. APRIL BTS RAW SOURCE")
print("-" * 72)

print("Source columns:", len(bts_source.columns))
print(
    "Required fields present:",
    len(required_fields.intersection(set(bts_source.columns)))
)

if len(bts_source.columns) != 110:
    print("FAIL: April BTS source should have 110 columns")
    all_passed = False

if not required_fields.issubset(set(bts_source.columns)):
    print("FAIL: required BTS fields are missing")
    all_passed = False
else:
    print("PASS: April BTS source contract")


print("\n4. TASK BOUNDARY")
print("-" * 72)

bronze_flight_rows = spark.table("brz_bts_flights").count()

if bronze_flight_rows == 0:
    print(
        "PASS: April flight data has NOT yet been ingested "
        "into Bronze."
    )
else:
    print(
        f"FAIL: brz_bts_flights already contains "
        f"{bronze_flight_rows} rows."
    )
    all_passed = False


print("\n" + "=" * 72)

if all_passed:
    print("TASK 8 ENVIRONMENT STATUS: READY")
else:
    raise RuntimeError(
        "Task 8 validation failed. Review the FAIL messages above."
    )

print("=" * 72)

StatementMeta(, 453adbe2-d7fb-429b-84e4-7be78494f99f, 11, Finished, Available, Finished, False)

AIROPS 360 - TASK 8 BRONZE ENVIRONMENT VALIDATION

1. DELTA TABLES
------------------------------------------------------------------------
brz_bts_flights              rows=0    columns=120
brz_weather_api_raw          rows=0    columns=12
brz_ingestion_audit          rows=0    columns=30

2. AIRPORT CONFIGURATION
------------------------------------------------------------------------
Airport rows: 15
Active airports: 15
PASS: airport configuration

3. APRIL BTS RAW SOURCE
------------------------------------------------------------------------
Source columns: 110
Required fields present: 20
PASS: April BTS source contract

4. TASK BOUNDARY
------------------------------------------------------------------------
PASS: April flight data has NOT yet been ingested into Bronze.

TASK 8 ENVIRONMENT STATUS: READY
